# Домашняя работа 6. Дерево и AdaBoost своими руками

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| К семинару | занятие 6 — Решающие деревья и композиции алгоритмов |
| Опора | материал семинара 6 и лекций до него |
| Ожидаемое время | 4 часа |

Две классические реализации. Сначала решающее дерево: рекурсивное построение с перебором порогов — код короткий, но требует аккуратности. Затем AdaBoost строго по теореме 5.15, с проверкой, что формула для $\alpha_t$ действительно доставляет минимум.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO`, код исполняется сверху вниз без ошибок
   в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами;
3. графики подписаны: заголовок, оси, легенда;
4. вариант ваш собственный (ячейка ниже).

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import optimize
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=6)
describe_variant(variant)

---
# Задача 1. Решающее дерево

Алгоритм: в вершине перебрать все признаки и все пороги, выбрать расщепление с
максимальным приростом, рекурсивно повторить для обеих частей. Остановка — по
глубине, по числу объектов или при нулевом приросте.

Порогами достаточно брать середины между соседними **различными** значениями
признака: между одинаковыми значениями разбиения не существует.

In [ ]:
def gini(y):
    """Gini(R) = 1 - sum p_k^2."""
    raise NotImplementedError


class Node:
    """Узел: либо лист со значением (вектор долей классов),
    либо предикат [x_feature <= threshold] с двумя потомками."""
    # TODO


class MyDecisionTree:
    """Решающее дерево для классификации (критерий Джини).

    __init__(max_depth, min_samples_leaf, min_samples_split)
    _best_split(X, y) -> (gain, feature, threshold): перебрать признаки и пороги
        (середины между соседними РАЗЛИЧНЫМИ значениями), вернуть лучший.
    _grow(X, y, depth): рекурсия; остановка по глубине, по числу объектов,
        при одном классе или при нулевом приросте.
    predict(X), n_leaves().
    """
    # TODO

### Задание 1.1. Сверка со `sklearn`

При одинаковых ограничениях ваше дерево обязано давать то же качество и то же
число листьев. Расхождение на 1–2 листа допустимо: при равных приростах порядок
перебора признаков может отличаться.

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=RANDOM_STATE)
Xa, Xv, ya, yv = train_test_split(Xm, ym, test_size=0.35, random_state=RANDOM_STATE)

# TODO: для max_depth in [1, 2, 3, 5, 8, 12, None] сравните своё дерево
#       и DecisionTreeClassifier(criterion="gini"): точность на контроле
#       и число листьев.

> **Вывод.** Совпало ли качество и число листьев? Где ваша реализация медленнее `sklearn` и почему?
>
> *(ваш ответ здесь)*

---
# Задача 2. AdaBoost по теореме 5.15

Теорема 5.15: при весах $w_i = \exp(-y_iF_{t-1}(x_i))$ и взвешенной ошибке
$\mathrm{err}_t$ оптимальный вес слабого классификатора равен

$$
\alpha_t = \frac12\ln\frac{1 - \mathrm{err}_t}{\mathrm{err}_t},
$$

после чего веса обновляются: $w_i \leftarrow w_i\exp(-\alpha_t y_i b_t(x_i))$
с последующей нормировкой.

In [ ]:
def adaboost_fit(X, y, n_estimators=200, seed=0):
    """AdaBoost для y из {-1, +1}. На каждом шаге:
       1) обучить пень DecisionTreeClassifier(max_depth=1) с весами w
          (параметр sample_weight);
       2) посчитать взвешенную ошибку err_t (долю веса на ошибочных объектах);
       3) alpha_t = 0.5 * ln((1 - err_t)/err_t)   -- теорема 5.15;
       4) w_i <- w_i exp(-alpha_t y_i b_t(x_i)), нормировать.
       Возвращает (список пней, массив alpha, историю по шагам).
    """
    raise NotImplementedError


def adaboost_predict(X, stumps, alphas):
    """sign(sum alpha_t b_t(x))."""
    raise NotImplementedError

### Задание 2.1. Проверка формулы и сверка со `sklearn`

Формула для $\alpha_t$ получена дифференцированием
$\sum_i w_ie^{-\alpha y_ib_t(x_i)}$ по $\alpha$. Проверьте её численно:
минимизируйте эту функцию по $\alpha$ и сравните с формулой.

In [ ]:
Xb, yb = make_moons(n_samples=400, noise=0.32, random_state=RANDOM_STATE)
yb = np.where(yb == 0, -1, 1)
Xba, Xbv, yba, ybv = train_test_split(Xb, yb, test_size=0.35, random_state=RANDOM_STATE)

# TODO: 1) обучите свой AdaBoost и сравните точность с AdaBoostClassifier;
#       2) для ПЕРВОГО шага (веса равномерные) сравните alpha по формуле
#          теоремы 5.15 с минимумом sum_i w_i exp(-alpha y_i b(x_i)),
#          найденным optimize.minimize_scalar.

In [ ]:
# TODO: постройте два графика: (а) ошибка композиции и экспоненциальная потеря
#       от T в логарифмическом масштабе; (б) err_t и alpha_t по шагам
#       с горизонталью на уровне 0.5.

> **Вывод.** Совпал ли $\alpha_t$ с численным минимумом? Как ведёт себя $\mathrm{err}_t$ с ростом $t$ и почему обучающая ошибка композиции падает до нуля, хотя каждый пень ошибается почти в 40 % случаев?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Почему у случайного леса можно ставить `n_estimators=1000` не задумываясь, а у градиентного бустинга — нельзя?
2. AdaBoost минимизирует экспоненциальную потерю $e^{-M}$. Сравните её с логистической $\ln(1+e^{-M})$ при $M \to -\infty$ и объясните, почему AdaBoost чувствителен к выбросам в разметке.

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.